# Tool Calling Patterns and Routing

This notebook shows the patterns that make tool-using agents reliable in practice.

The key idea is simple: the model should choose between small tools, not invent the whole workflow.


## What good tool design looks like

- each tool has one clear job
- each tool has typed inputs and outputs
- each tool has a timeout and retry policy
- the model can see tool results, not hidden side effects
- the system can route to a fallback if a tool fails


In [ ]:
from dataclasses import dataclass
from typing import Callable, Any

@dataclass
class ToolSpec:
    name: str
    description: str
    func: Callable[[dict[str, Any]], Any]

def lookup_customer(payload: dict[str, Any]) -> dict[str, Any]:
    customer_id = payload.get('customer_id', 'unknown')
    return {'customer_id': customer_id, 'segment': 'premium', 'status': 'active'}

def fetch_policy(payload: dict[str, Any]) -> dict[str, Any]:
    return {'policy_name': payload.get('policy_name', 'default'), 'answer': 'approved if confidence > 0.8'}

tools = [
    ToolSpec('lookup_customer', 'Get customer profile and segment', lookup_customer),
    ToolSpec('fetch_policy', 'Read policy guidance for the request', fetch_policy),
]

[tool.name for tool in tools]


In [ ]:
def route_tool(question: str) -> str:
    q = question.lower()
    if 'customer' in q or 'segment' in q:
        return 'lookup_customer'
    if 'policy' in q or 'rule' in q:
        return 'fetch_policy'
    return 'fetch_policy'

route_tool('Show the customer segment and next best action')


## Validation before execution

A strong pattern is to validate tool input before the tool is called. That keeps bad requests from becoming downstream failures.


In [ ]:
def validate_payload(tool_name: str, payload: dict[str, Any]) -> list[str]:
    errors = []
    if tool_name == 'lookup_customer' and 'customer_id' not in payload:
        errors.append('customer_id is required')
    if tool_name == 'fetch_policy' and 'policy_name' not in payload:
        errors.append('policy_name is required')
    return errors

validate_payload('lookup_customer', {'customer_id': 42})


## LangChain and LangGraph tips

- Use LangChain for the tool wrapper and message plumbing
- Use LangGraph for explicit routing, retries, and approval nodes
- Keep the routing logic separate from the tool implementation
- Make the failure path first-class, not an afterthought


In [ ]:
def run_tool(tool_name: str, payload: dict[str, Any]) -> Any:
    spec = next(tool for tool in tools if tool.name == tool_name)
    errors = validate_payload(tool_name, payload)
    if errors:
        return {'status': 'error', 'errors': errors}
    return {'status': 'ok', 'result': spec.func(payload)}

run_tool('lookup_customer', {'customer_id': 101})


## Useful production tricks

- log the chosen tool and payload shape
- add a circuit breaker if a tool starts failing repeatedly
- return a typed result so downstream code can branch safely
- separate read tools from write tools
- if the model is uncertain, ask a clarifying question instead of guessing


## Demo ideas

Good examples for this notebook:

- customer segment lookup
- policy check for a workflow decision
- routing to the right data source
- fallback when a tool returns an error
